In this one, in the end I show a p value that shows that there is more correlaiton for the last digits

In [10]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr 

###############################################################################
# Paths & constants
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")

FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

# channels you want to **keep**
KEEP = {'AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','C1','C2'}          # 13 chans

weights = np.array([1.0 if ch in KEEP else 0.0 for ch in FRONTAL_MIDLINE])
weights = weights / np.linalg.norm(weights)     # optional normalisation

OUT_TRIALS  = Path("trial_level_cca_fixedlag.csv")
OUT_SUBJECT = Path("subject_best_lag.csv")

SUBJECTS = np.setdiff1d(np.arange(32, 99), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
#SUBJECTS = np.setdiff1d(np.arange(32, 99), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
SHIFTS = np.arange(-100, 101)      # ±1 s at 100 Hz → samples
WIN_OFFSET1 = 200                  # discard first 200 ms
WIN_OFFSET2 = 110                  # discard last 110 ms

###############################################################################
# Helper functions
###############################################################################

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over time×channels."""
    x = x - x.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.mean(x**2))
    return x / scale


def cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> float:
    """Canonical correlation (single component)."""
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(eeg, pupil)
    u, v = cca.transform(eeg, pupil)
    return float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])


In [2]:
def load_all_trials(
        sub: int,
        eeg_root: Path = EEG_ROOT,
        pupil_root: Path = PUPIL_ROOT,
        min_len: int = 40,
        max_len_diff: int = 30,
) -> list[tuple[np.ndarray, np.ndarray, dict]]:
    """
    Load *all* valid EEG-pupil trial pairs for one subject.

    Parameters
    ----------
    sub : int
        Numeric subject ID (e.g. 42).
    eeg_root, pupil_root : Path
        Roots of the pre-processed EEG and pupil folders.
    min_len : int
        Minimum number of samples a pupil trace must have to be accepted.
    max_len_diff : int
        Reject trial if |len(pupil)-len(eeg)| exceeds this.
    Returns
    -------
    trials : list of (eeg, pupil_z, meta)
        * eeg        – (T × n_channels) float64, already centred/scaled
        * pupil_z    – (T × 1) float64, per-trial z-scored
        * meta       – dict with subject/condition/load/epoch
    """
    trials = []
    sub_tag = f"sub-{sub:03d}"
    eeg_sub  = eeg_root   / sub_tag
    pupil_sub = pupil_root / sub_tag

    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        return trials

    # iterate condition (“control” / “memory”) and load (“05” / “09” / “13”)
    for cond_path in sorted(eeg_sub.iterdir()):
        if not cond_path.is_dir():
            continue
        for load_path in sorted(cond_path.iterdir()):
            if not load_path.is_dir():
                continue

            # matching pupil directory
            pupil_path = pupil_sub / cond_path.name / load_path.name
            if not pupil_path.exists():
                continue

            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))
            common = {f.name for f in eeg_epochs} & {f.name for f in pupil_epochs}
            if not common:
                continue

            for fname in sorted(common):
                eeg_df = pd.read_csv(load_path / fname, comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#",
                                       names=["time", "diameter_z"], index_col=0)

                eeg   = eeg_df.values.astype(float)
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic validity checks
                if len(pupil) < min_len or abs(len(pupil) - len(eeg)) > max_len_diff:
                    continue

                # normalise signals ----------------------------------------
                eeg_norm = normalise_eeg(eeg)           # your helper from before
                pupil_z  = ((pupil - pupil.mean()) / pupil.std(ddof=0))

                # same number of samples
                T = min(len(eeg_norm), len(pupil_z))
                eeg_norm = eeg_norm[0:T, :]  # (T × n_channels)
                pupil_z  = pupil_z[0:T].reshape(-1, 1)

                meta = {
                    "subject":   sub_tag,
                    "condition": cond_path.name,
                    "load":      int(load_path.name),
                    "epoch":     fname
                }
                trials.append((eeg_norm, pupil_z, meta))

    return trials

from typing import List, Tuple
import numpy as np

def split_trials_by_condition(
        trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        memory_label: str = "memory",
        control_label: str = "control"
) -> Tuple[List[Tuple[np.ndarray, np.ndarray, dict]], List[Tuple[np.ndarray, np.ndarray, dict]]]:
    """
    Separate a mixed list of (eeg, pupil, meta) trial tuples into memory-condition and control-condition sub-lists.

    Parameters
    ----------
    trials : list of tuples
        Each tuple = (eeg_array, pupil_array, meta_dict).
        meta_dict must contain a key 'condition'.
    memory_label : str
        The value of meta['condition'] that marks a memory trial.
    control_label : str
        The value of meta['condition'] that marks a control trial.

    Returns
    -------
    memory_trials  : list[tuple]
    control_trials : list[tuple]
    """
    memory_trials  = []
    control_trials = []

    for eeg, pupil, meta in trials:
        cond = meta.get("condition", "").lower()
        if cond == memory_label:
            memory_trials.append((eeg, pupil, meta))
        elif cond == control_label:
            control_trials.append((eeg, pupil, meta))
        else: raise ValueError(f"Unknown condition label: {cond}")

    return memory_trials, control_trials


In [3]:
import numpy as np
from typing import List, Tuple, Dict

def search_best_lag(
        train_trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        shifts: np.ndarray = SHIFTS,
        return_curve: bool = False
) -> Tuple[float, int, Dict[int, float] | None]:
    """
    Search for the lag (sample shift) that maximises the mean canonical
    correlation between EEG and pupil traces in a training set.

    Parameters
    ----------
    train_trials : list of (eeg, pupil_z, meta)
        Each eeg  : 2-D array [time × channels or CCA-components]
        Each pupil: 1-D array [time]
    shifts : np.ndarray
        Array of integer lag shifts (positive = EEG is moved forward).
    return_curve : bool, default False
        If True, also return the full {shift: mean_r} dictionary.

    Returns
    -------
    best_corr : float
        Highest mean canonical correlation found.
    best_shift : int
        Shift (samples) that maximised the correlation.
    mean_r_per_shift : dict | None
        Only when `return_curve` is True.
    """
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)         # common window
    mean_r_per_shift: Dict[int, float] = {}

    for s in shifts:                               # <-- loop over *shifts*, not SHIFTS
        rs = []
        for eeg, pupil_z, _ in train_trials:
            eeg_shifted = np.roll(eeg, s, axis=0)[win]
            r = cca_corr(eeg_shifted, pupil_z[win])
            rs.append(r)
        mean_r_per_shift[s] = float(np.mean(rs))   # cast to plain float for JSON-ability

    best_shift = max(mean_r_per_shift, key=mean_r_per_shift.get)
    best_corr  = mean_r_per_shift[best_shift]

    if return_curve:
        return best_corr, best_shift, mean_r_per_shift
    else:
        return best_corr, best_shift, None


In [4]:

from typing import List, Tuple, Optional

# trials  : list of (eeg, pupil_z, meta)   –– the tuples returned by load_all_trials
# shift   : integer sample shift (best_shift)
# win     : slice or None                  –– cropping window (set to None if the
#                                            trials are already pre-trimmed)
def concat_trials(
        trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        shift: int = 0,
        win: Optional[slice] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Concatenate a list of trials into one long matrix pair ready for CCA.

    Returns
    -------
    X  : ndarray, shape (Σ Tᵢ, n_channels)
    Y  : ndarray, shape (Σ Tᵢ, 1)
    """
    X_blocks, Y_blocks = [], []

    for eeg, pupil_z, _ in trials:
        # 1. roll *inside* the trial so samples never cross trial boundaries
        eeg_shift = np.roll(eeg, shift, axis=0)

        # 2. optional windowing (do it once here if you did **not** crop in loader)
        if win is not None:
            eeg_shift = eeg_shift[win]
            pupil_seg = pupil_z[win]
        else:
            pupil_seg = pupil_z          # already trimmed earlier

        # 3. stack
        X_blocks.append(eeg_shift)
        Y_blocks.append(pupil_seg)

    X = np.vstack(X_blocks)
    Y = np.vstack(Y_blocks)
    return X, Y

def iterate_trials(trials, shift, win):
    """Yield (eeg_shifted, pupil_z_windowed, meta) one by one."""
    for eeg, pupil_z, meta in trials:
        eeg_s = np.roll(eeg, shift, axis=0)[win]
        yield eeg_s, pupil_z[win], meta.copy()

import pandas as pd
import json
from pathlib import Path

def save_cca_weights(cca, subject_tag, lag_ms, eeg_ch_names, condition, out_dir=Path("weights2")):
    """
    Dump EEG & pupil canonical weights to CSV/JSON for one subject.

    Parameters
    ----------
    cca            : fitted sklearn.cross_decomposition.CCA
    subject_tag    : "sub-042"
    lag_ms         : e.g. -90
    eeg_ch_names   : list[str] same order as columns in your trial matrices
    out_dir        : destination folder (created if missing)
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # ------- Pupil weight -> JSON ---------------------------------------
    w_pupil = float(cca.y_weights_[0, 0])   # scalar in 1-dim pupil case
    w_pupil = w_pupil / w_pupil
    with open(out_dir / f"{subject_tag}_pupil_weight_{condition}_{lag_ms:+d}ms.json", "w") as fh:
        json.dump({"weight": w_pupil}, fh, indent=2)
    
    # ------- EEG weights -> tidy CSV ------------------------------------
    w_eeg = cca.x_weights_[:, 0] / w_pupil
    w_eeg = pd.Series(w_eeg, index=eeg_ch_names, name="weight")
    w_eeg.index.name = "channel"
    w_eeg.to_csv(out_dir / f"{subject_tag}_eeg_weights_{condition}_{lag_ms:+d}ms.csv")

    print(f"saved weights for {subject_tag} (lag {lag_ms:+d} ms)")



In [5]:
best_shift_by_sub = {33: -5, 34: 32, 35: -8, 36: -59, 38: -1, 39: 2, 40: 100, 41: -10, 42: -80, 43: 70, 44: -10, 45: 42, 46: -68, 47: 100, 48: -63, 49: 46, 50: 91, 51: 100, 52: 81, 54: 63, 55: -38, 56: 23, 57: 36, 58: 46, 59: -7, 60: 100, 62: -9, 63: 0, 64: 53, 65: -6, 67: 2, 68: 96, 69: 28, 70: 16, 71: 74, 72: 0, 73: 38, 74: -100, 75: 32, 76: 100, 77: 76, 79: 80, 80: -1, 81: 64, 82: -44, 83: -52, 85: 37, 86: 100, 87: 24, 88: 100, 89: -100, 91: -5, 92: 41, 93: 100, 95: -7, 97: -56, 98: 69}
print(best_shift_by_sub)

{33: -5, 34: 32, 35: -8, 36: -59, 38: -1, 39: 2, 40: 100, 41: -10, 42: -80, 43: 70, 44: -10, 45: 42, 46: -68, 47: 100, 48: -63, 49: 46, 50: 91, 51: 100, 52: 81, 54: 63, 55: -38, 56: 23, 57: 36, 58: 46, 59: -7, 60: 100, 62: -9, 63: 0, 64: 53, 65: -6, 67: 2, 68: 96, 69: 28, 70: 16, 71: 74, 72: 0, 73: 38, 74: -100, 75: 32, 76: 100, 77: 76, 79: 80, 80: -1, 81: 64, 82: -44, 83: -52, 85: 37, 86: 100, 87: 24, 88: 100, 89: -100, 91: -5, 92: 41, 93: 100, 95: -7, 97: -56, 98: 69}


In [12]:
import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
DIG_LEN   = 200           # 2 s at 100 Hz
rows_trials = []
rows_subject = []  

for subj in SUBJECTS:
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)
    """
    # ----- lag search on memory ------------------------------------------
    return_curve = False  # set to True if you want the full curve
    best_corr, best_shift, mean_r_per_shift = search_best_lag(trials_memory, SHIFTS, return_curve)
    print(f"Subject {subj:02d}: best lag {best_shift*10} ms with r = {best_corr:.3f}")
    best_shift_by_sub[subj] = best_shift      # samples, not ms

    if return_curve:
        # -------------------------------------------------------------
        # 2) Build x and y vectors for plotting
        # -------------------------------------------------------------
        lags   = np.array(sorted(mean_r_per_shift))                 # x-axis (samples)
        r_mean = np.array([mean_r_per_shift[s] for s in lags])      # y-axis (mean r)

        # If you prefer milliseconds instead of samples:
        # fs = 1000  # replace with your real sampling rate
        # lags = lags * 1000 / fs

        # -------------------------------------------------------------
        # 3) Plot
        # -------------------------------------------------------------
        import matplotlib.pyplot as plt

        plt.figure(figsize=(6, 3.5))
        plt.plot(lags, r_mean, lw=2)
        plt.axvline(best_shift, ls='--', lw=1.5,
                    label=f'best lag = {best_shift}')
        plt.xlabel('Lag (samples)')
        plt.ylabel('Mean canonical correlation (r)')
        plt.title('Mean CCA correlation vs. lag')
        plt.grid(alpha=.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

"""
    best_shift = best_shift_by_sub[subj] if subj in best_shift_by_sub else print("MISTAKE")
    # ----- fit weights ONCE using all train trials at best_shift ---------
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    best_shift = int(best_shift)  # convert to int if it was float
    X_mem, Y_mem = concat_trials(trials_memory, shift=best_shift, win=win)
    X_ctrl, Y_ctrl = concat_trials(trials_control, shift=best_shift, win=win)
    
    cca_mem = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    # cca_mem.fit(X_mem, Y_mem)

    cca_ctrl = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    # cca_ctrl.fit(X_ctrl, Y_ctrl)

    
    # -------------------------------------------------------------
    # 2) whole‑trial correlation for MEMORY
    # -------------------------------------------------------------
    # cvX_m, cvY_m = cca_mem.transform(X_mem, Y_mem)
    # r_mem, p_mem = pearsonr(cvX_m[:, 0], cvY_m[:, 0])

    eeg_mem = X_mem @ weights              # (samples,)   weighted theta power
    pup_mem = Y_mem.squeeze()        # (samples,)   pupil diameter

    # Pearson correlation
    r_mem, p_mem = pearsonr(eeg_mem, pup_mem)

    rows_subject.append({
        "subject":   subj,
        "condition": "memory",
        "lag_ms":    best_shift * 10,
        "r":         float(r_mem),
        "p-value": float(p_mem),
    })

    # -------------------------------------------------------------
    # 3) whole‑trial correlation for CONTROL
    # -------------------------------------------------------------
    # cvX_c, cvY_c = cca_ctrl.transform(X_ctrl, Y_ctrl)
    # r_ctrl, p_ctrl = pearsonr(cvX_c[:, 0], cvY_c[:, 0])
    eeg_ctl = X_ctrl @ weights
    pup_ctl = Y_ctrl.squeeze()
    r_ctrl, p_ctrl = pearsonr(eeg_ctl, pup_ctl)


    rows_subject.append({
        "subject":   subj,
        "condition": "control",
        "lag_ms":    best_shift * 10,
        "r":         float(r_ctrl),
        "p-value": float(p_ctrl), 
    })

    # save_cca_weights(cca_mem, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE)
    # save_cca_weights(cca_ctrl, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "ctrl", eeg_ch_names=FRONTAL_MIDLINE)

    # ----- per-trial r on TEST with frozen lag & weights -----------------
    """
    for eeg_s, pupil_s, meta in iterate_trials(trials_memory, best_shift, win):
        n_digits = meta['load']
        for d in range(n_digits):
            w = slice(d*DIG_LEN, (d+1)*DIG_LEN)

        #cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
        #cca.fit(eeg_s, pupil_s)
        cv_eeg, cv_pupil = cca_mem.transform(eeg_s, pupil_s)
        cv_eeg = cv_eeg[:, 0]  # first CCA component
        cv_pupil = cv_pupil[:, 0]  # first CCA component

        for d in range(n_digits):
            w = slice(d * DIG_LEN, (d + 1) * DIG_LEN)
            x, y = cv_eeg[w], cv_pupil[w]
            
            if x.size == 0 or y.size == 0:
                print("Empty slice:", meta['subject'], meta['epoch'], d)
                continue

            if not np.isfinite(x).all() or not np.isfinite(y).all():
                print("Non-finite values:", meta['subject'], meta['epoch'], d, "→", np.sum(~np.isfinite(x)), "bad in X;", np.sum(~np.isfinite(y)), "bad in Y")
                continue

            if np.std(x) == 0 or np.std(y) == 0:
                print("Const window:", meta['subject'], meta['epoch'], d, np.std(x), np.std(y))
                continue

            r = np.corrcoef(x, y)[0, 1]
            rows_trials.append({
                'subject': meta['subject'],
                'condition': meta['condition'],
                'load': meta['load'],
                'epoch': meta['epoch'],
                'lag_ms': best_shift * 10,  # convert samples to ms
                'digit_pos': d + 1,  # 1-based digit index
                'r': float(r),  # convert to plain float for JSON-ability
            })

        meta['r'] = float(np.corrcoef(cv_eeg, cv_pupil)[0,1])
        meta['lag_ms'] = best_shift *10
        all_rows.append(meta)
        """

df_sameW = pd.DataFrame(rows_trials)
grand = (df_sameW.query("condition == 'memory'")
           .groupby(['load','digit_pos'])['r']
           .agg(['mean','sem'])
           .reset_index())

for L in [5,9,13]:
    sub = grand[grand.load == L]
    plt.errorbar(sub.digit_pos, sub['mean'],
                 yerr=sub['sem'], label=f"{L} digits")
plt.axhline(0, c='k', lw=.5)
plt.xlabel("Digit position in the sequence")
plt.ylabel("CCA correlation (mean ± SEM)")
plt.legend(); plt.tight_layout()

##############################################################################
# ---- save -----------------------------------------------------------------
##############################################################################
# pd.DataFrame(all_rows).to_csv("trial_level_cca_fixedlag_myW.csv", index=False)
# print("Finished - saved per-trial correlations with plain CCA.")

##############################################################################
# 4)  after the loop – save per‑subject correlations
##############################################################################
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlagmyW.csv", index=False)
print("Finished - saved subject-level correlations as subject_level_cca_fixedlag.csv.")


Processing subject 33...
Processing subject 34...
Processing subject 35...
Processing subject 36...
Processing subject 38...
Processing subject 39...
Processing subject 40...
Processing subject 41...
Processing subject 42...
Processing subject 43...
Processing subject 44...
Processing subject 45...
Processing subject 46...
Processing subject 47...
Processing subject 48...
Processing subject 49...
Processing subject 50...
Processing subject 51...
Processing subject 52...
Processing subject 54...
Processing subject 55...
Processing subject 56...
Processing subject 57...
Processing subject 58...
Processing subject 59...
Processing subject 60...
Processing subject 62...
Processing subject 63...
Processing subject 64...
Processing subject 65...
Processing subject 67...
Processing subject 68...
Processing subject 69...
Processing subject 70...
Processing subject 71...
Processing subject 72...
Processing subject 73...
Processing subject 74...
Processing subject 75...
Processing subject 76...


UndefinedVariableError: name 'condition' is not defined

In [13]:
##############################################################################
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlagmyW.csv", index=False)
print("Finished - saved subject-level correlations as subject_level_cca_fixedlag.csv.")

Finished - saved subject-level correlations as subject_level_cca_fixedlag.csv.


In [ ]:
import pandas as pd
from scipy.stats import ttest_rel

# -------------------------------------------------------------------
# 1)  Load the file  (keep the raw string r'' so backslashes work)
# -------------------------------------------------------------------
csv_path = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\subject_level_cca_fixedlagmyW.csv"
df = pd.read_csv(csv_path)

# -------------------------------------------------------------------
# 2)  Pivot: one row per subject, two columns ('memory', 'control')
# -------------------------------------------------------------------
pivot = (df
         .pivot(index='subject', columns='condition', values='r')
         .dropna(subset=['memory', 'control']))   # drop subjects missing a condition, if any

# -------------------------------------------------------------------
# 3)  Paired t-test  (SciPy gives a two-tailed p; convert to one-tailed)
# -------------------------------------------------------------------
t_stat, p_two_sided = ttest_rel(pivot['memory'], pivot['control'])

p_one_sided = p_two_sided / 2 if t_stat > 0 else 1 - p_two_sided / 2

# -------------------------------------------------------------------
# 4)  Output
# -------------------------------------------------------------------
print(f"Subjects included: {len(pivot)}")
print(f"Paired t-statistic: {t_stat:.4f}")
print(f"One-sided p-value (memory > control): {p_one_sided:.6g}")


Subjects included: 57
Paired t-statistic: 3.5750
One-sided p-value (memory > control): 0.000364993


: 